<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/nemotron_bolt_e4b_fineweb_topo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!ls -tlta /content/drive/MyDrive/topogpt/

total 60
drwx------ 15 root root 4096 Oct 24 01:06 .
drwx------ 47 root root 4096 Oct 23 20:24 ..
drwx------  2 root root 4096 Oct 23 12:01 muse_glimmer_30b_fineweb_topo
drwx------  2 root root 4096 Oct 19 22:54 sarvam_run_new
drwx------  2 root root 4096 Oct 19 20:44 sarvam_topo_run_new
drwx------  2 root root 4096 Oct 19 19:52 hf_cache
drwx------  2 root root 4096 Oct 18 10:46 sarvam_run_french
drwx------  2 root root 4096 Oct 17 19:14 sarvam_topo_run
drwx------  2 root root 4096 Oct 17 15:48 sarvam_run
drwx------  2 root root 4096 Oct 15 10:02 deepseek-r1-distill-llama-70b-topocbp-nolora
drwx------  2 root root 4096 Oct 15 01:11 mixtral-8x22b-topocbp-nolora
drwx------  2 root root 4096 Oct 12 20:02 riemann_mendeleev_70b_run
drwx------  2 root root 4096 Oct 12 14:15 llama_70b_rh_demo_run
drwx------  2 root root 4096 Oct 11 11:17 gpt_oss_20b_rh_demo_run
drwx------  2 root root 4096 Oct  9 10:47 gpt_oss_20b_hf_topo_run


In [4]:
!rm -rf /content/drive/MyDrive/topogpt/nemotron_bolt_e4b_fineweb_topo

In [3]:
!pip install -q -U bitsandbytes

In [1]:
!nvidia-smi

Sat Oct 24 02:39:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   61C    P0             23W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## TRAIN

In [2]:
import os
import gc
import hashlib
import json
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Optimizer
from torch.utils.data import Dataset
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    logging as hf_logging
)

# Prevent CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

hf_logging.set_verbosity_error()

# =====================================================================
# 1. GLOBAL CONFIGURATION & REPRODUCIBILITY SEED
# =====================================================================
SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

MODEL_ID = 'nvidia/Nemotron-3.5-Bolt-E4B'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]
SAFETY_CONSTANT = 0.9785142874

OUTPUT_DIR = "/content/drive/MyDrive/topogpt/nemotron_bolt_e4b_fineweb_topo"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FINEWEB_SAMPLES = 5000
MAX_SEQ_LENGTH = 256

# =====================================================================
# 2. TOPO-CBP CUSTOM OPTIMIZER CLASS (GOVERNS LM_HEAD DIRECTLY)
# =====================================================================
class TopoAdamW(Optimizer):
    """
    Topological Continual Backpropagation (Topo-CBP) Custom Optimizer.
    Governs the active trainable lm_head directly to enforce zero prime drift.
    """
    def __init__(self, params, lr=1e-5, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=1e-2, amsgrad=False, governed_layer=None,
                 prime_anchors=None, lambda_euler=SAFETY_CONSTANT):
        if governed_layer is None:
            raise ValueError("Topological Governor requires 'governed_layer' (lm_head) to be explicitly passed.")

        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay, amsgrad=amsgrad)
        super(TopoAdamW, self).__init__(params, defaults)

        self.governed_layer = governed_layer
        self.prime_anchors = prime_anchors or PRIME_ANCHORS
        self.lambda_euler = lambda_euler

        weight_data = self.governed_layer.weight.data
        vocab_size = weight_data.shape[0]
        self.valid_anchors = [idx for idx in self.prime_anchors if idx < vocab_size]

        # Bedrock Mapping: Lock the exact initial state of lm_head rows
        self.baseline_invariant_tensor_map = {
            idx: weight_data[idx].detach().clone() for idx in self.valid_anchors
        }

        # Certified Baseline Hardware Cryptographic Digest
        serialized = torch.cat([t.flatten().float() for t in self.baseline_invariant_tensor_map.values()]).cpu().numpy().tobytes()
        self.certified_sha256_digest = hashlib.sha256(serialized).hexdigest()
        print(f"[TopoAdamW] Active lm_head Bedrock Locked. Certified Digest: {self.certified_sha256_digest[:16]}")

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        # GOVERNOR PHASE I: Pre-Step Gradient Clamping on lm_head (∇_P L = 0)
        if self.governed_layer.weight.grad is not None:
            grad = self.governed_layer.weight.grad
            grad.mul_(self.lambda_euler)
            for idx in self.valid_anchors:
                grad[idx].zero_()

        # Execute AdamW updates on plastic parameters
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue

                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    if group['amsgrad']:
                        state['max_exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)

                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                if group['amsgrad']:
                    max_exp_avg_sq = state['max_exp_avg_sq']
                beta1, beta2 = group['betas']

                state['step'] += 1

                # Decoupled weight decay
                if group['weight_decay'] != 0:
                    p.data.mul_(1.0 - group['lr'] * group['weight_decay'])

                exp_avg.mul_(beta1).add_(p.grad, alpha=1.0 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(p.grad, p.grad, value=1.0 - beta2)

                bias_correction1 = 1.0 - beta1 ** state['step']
                bias_correction2 = 1.0 - beta2 ** state['step']

                if group['amsgrad']:
                    torch.max(max_exp_avg_sq, exp_avg_sq, out=max_exp_avg_sq)
                    denom = (max_exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(group['eps'])
                else:
                    denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(group['eps'])

                step_size = group['lr'] / bias_correction1
                p.addcdiv_(exp_avg, denom, value=-step_size)

        # GOVERNOR PHASE II: Post-Step Bitwise Silicon Lock on lm_head
        weight_data = self.governed_layer.weight.data
        for idx, original_tensor in self.baseline_invariant_tensor_map.items():
            weight_data[idx].copy_(original_tensor)

        return loss

    def audit_telemetry(self):
        weight_data = self.governed_layer.weight.data
        max_drift = 0.0
        for idx, original_tensor in self.baseline_invariant_tensor_map.items():
            current_tensor = weight_data[idx]
            drift = torch.max(torch.abs(current_tensor - original_tensor)).item()
            if drift > max_drift:
                max_drift = drift

        snapshot_tensors = [weight_data[idx] for idx in self.valid_anchors]
        serialized = torch.cat([t.flatten().float() for t in snapshot_tensors]).detach().cpu().numpy().tobytes()
        current_hash = hashlib.sha256(serialized).hexdigest()

        hash_matched = (current_hash == self.certified_sha256_digest)
        return max_drift, current_hash, hash_matched

# =====================================================================
# 3. TASK MODEL WRAPPER
# =====================================================================
class NemotronFineWebModel(nn.Module):
    def __init__(self, base_model: nn.Module, vocab_size: int, embedding_layer: nn.Module, device: torch.device = DEVICE):
        super().__init__()
        self.base_model = base_model
        self.device = device

        self.hidden_size = getattr(base_model.config, "hidden_size", None)
        if self.hidden_size is None:
            self.hidden_size = getattr(getattr(base_model.config, "text_config", None), "hidden_size", 2560)

        # Final RMSNorm
        if hasattr(base_model, "model") and hasattr(base_model.model, "norm"):
            self.final_norm = base_model.model.norm
        elif hasattr(base_model, "norm"):
            self.final_norm = base_model.norm
        else:
            self.final_norm = nn.Identity()

        # Trainable projection head
        self.lm_head = nn.Linear(self.hidden_size, vocab_size, bias=False, dtype=torch.bfloat16).to(self.device)

        # Initialize from base embedding weights
        print("[NemotronFineWebModel] Seeding trainable lm_head from base embeddings...")
        with torch.no_grad():
            self.lm_head.weight.copy_(embedding_layer.weight.data.to(torch.bfloat16))

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True,
        )

        hidden_states = self.final_norm(outputs.hidden_states[-1])
        actual_hidden_dim = hidden_states.size(-1)

        if labels is None:
            return self.lm_head(hidden_states)

        shift_hidden = hidden_states[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        flat_hidden = shift_hidden.view(-1, actual_hidden_dim)
        flat_labels = shift_labels.view(-1)

        valid_mask = flat_labels != -100
        active_hidden = flat_hidden[valid_mask]
        active_labels = flat_labels[valid_mask]

        total_loss = torch.tensor(0.0, device=self.device, dtype=torch.float32)
        total_tokens = active_labels.numel()

        if total_tokens > 0:
            chunk_size = 128
            for i in range(0, total_tokens, chunk_size):
                chunk_h = active_hidden[i:i + chunk_size]
                chunk_l = active_labels[i:i + chunk_size]
                chunk_logits = self.lm_head(chunk_h).float()
                loss_chunk = F.cross_entropy(chunk_logits, chunk_l, reduction='sum')
                total_loss = total_loss + loss_chunk

            loss = (total_loss / total_tokens).to(torch.bfloat16)
        else:
            loss = torch.tensor(0.0, device=self.device, dtype=torch.bfloat16, requires_grad=True)

        return {"loss": loss}

# =====================================================================
# 4. CUSTOM TRAINER BINDING TO LM_HEAD
# =====================================================================
class TopoTrainer(Trainer):
    def __init__(self, *args, governed_layer=None, prime_anchors=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.governed_layer = governed_layer
        self.prime_anchors = prime_anchors or PRIME_ANCHORS

    def create_optimizer(self):
        if self.optimizer is None:
            decay_parameters = self.get_decay_parameter_names(self.model)

            optimizer_grouped_parameters = [
                {
                    "params": [
                        p for n, p in self.model.named_parameters()
                        if (n in decay_parameters and p.requires_grad)
                    ],
                    "weight_decay": self.args.weight_decay,
                },
                {
                    "params": [
                        p for n, p in self.model.named_parameters()
                        if (n not in decay_parameters and p.requires_grad)
                    ],
                    "weight_decay": 0.0,
                },
            ]

            self.optimizer = TopoAdamW(
                optimizer_grouped_parameters,
                lr=self.args.learning_rate,
                betas=(self.args.adam_beta1, self.args.adam_beta2),
                eps=self.args.adam_epsilon,
                weight_decay=self.args.weight_decay,
                governed_layer=self.governed_layer,  # Points directly to model.lm_head
                prime_anchors=self.prime_anchors,
                lambda_euler=SAFETY_CONSTANT
            )

        return self.optimizer

class TopoTelemetryCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        optimizer = kwargs.get("optimizer")
        if hasattr(optimizer, "audit_telemetry"):
            max_drift, digest, hash_matched = optimizer.audit_telemetry()
            status = "100% PASS" if hash_matched and max_drift == 0.0 else "VIOLATION"
            print(f"\n[TOPO Telemetry] Epoch {state.epoch:.0f} | Drift: {max_drift:.12f} | Digest: {digest[:16]} | Status: {status}")
            if status != "100% PASS":
                raise RuntimeError("Integrity violation: Baseline hardware signature modified.")

# =====================================================================
# 5. DATASET PREPARATION
# =====================================================================
class FineWebDataset(Dataset):
    def __init__(self, hf_stream, tokenizer, num_samples=FINEWEB_SAMPLES, max_length=MAX_SEQ_LENGTH):
        self.samples = []
        count = 0
        print(f"Buffering {num_samples} sequences from FineWeb stream...")
        for row in hf_stream:
            text = row.get("text", "")
            if len(text.strip()) < 50:
                continue
            enc = tokenizer(
                text,
                max_length=max_length,
                truncation=True,
                padding=False,
                return_tensors=None
            )
            if len(enc["input_ids"]) > 16:
                self.samples.append({
                    "input_ids": enc["input_ids"],
                    "attention_mask": enc["attention_mask"]
                })
                count += 1
                if count >= num_samples:
                    break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# =====================================================================
# 6. EXECUTION HARNESS
# =====================================================================
if __name__ == "__main__":
    print(f"Loading {MODEL_ID} in 4-bit NF4 precision for 4GB edge-footprint constraints...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_enable_fp32_cpu_offload=True
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        trust_remote_code=True
    )
    base_model.config.use_cache = False
    base_model.gradient_checkpointing_enable()

    # Freeze base model
    for param in base_model.parameters():
        param.requires_grad = False

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Retrieve base embedding layer to seed lm_head
    if hasattr(base_model, "get_input_embeddings"):
        embedding_layer = base_model.get_input_embeddings()
    elif hasattr(base_model, "model") and hasattr(base_model.model, "embed_tokens"):
        embedding_layer = base_model.model.embed_tokens
    else:
        embedding_layer = next(m for m in base_model.modules() if isinstance(m, torch.nn.Embedding))

    # Initialize model wrapper
    model = NemotronFineWebModel(base_model, tokenizer.vocab_size, embedding_layer, DEVICE)

    print("Loading 'HuggingFaceFW/fineweb' (sample-10BT)...")
    fineweb_stream = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=True)
    train_dataset = FineWebDataset(fineweb_stream, tokenizer, num_samples=FINEWEB_SAMPLES, max_length=MAX_SEQ_LENGTH)

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-5,
        weight_decay=0.01,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        use_cpu=(DEVICE.type == "cpu"),
        remove_unused_columns=False,
        bf16=True
    )

    # Pass model.lm_head directly as governed_layer
    trainer = TopoTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=data_collator,
        governed_layer=model.lm_head,
        prime_anchors=PRIME_ANCHORS,
        callbacks=[TopoTelemetryCallback()]
    )

    print("Beginning Topo-CBP continual training loop on FineWeb...")
    trainer.train()

    print(f"Persisting lm_head weights and tokenizer to: {OUTPUT_DIR}")
    torch.save(model.lm_head.state_dict(), os.path.join(OUTPUT_DIR, "lm_head.pt"))
    tokenizer.save_pretrained(OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading nvidia/Nemotron-3.5-Bolt-E4B in 4-bit NF4 precision for 4GB edge-footprint constraints...


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

[NemotronFineWebModel] Seeding trainable lm_head from base embeddings...
Loading 'HuggingFaceFW/fineweb' (sample-10BT)...


Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Buffering 5000 sequences from FineWeb stream...
Beginning Topo-CBP continual training loop on FineWeb...
[TopoAdamW] Active lm_head Bedrock Locked. Certified Digest: 360af91d41ed5e98
{'loss': '10.85', 'grad_norm': '11.75', 'learning_rate': '9.856e-06', 'epoch': '0.016'}
{'loss': '10.81', 'grad_norm': '8.625', 'learning_rate': '9.696e-06', 'epoch': '0.032'}
{'loss': '10.56', 'grad_norm': '9.875', 'learning_rate': '9.536e-06', 'epoch': '0.048'}
{'loss': '10.53', 'grad_norm': '8.625', 'learning_rate': '9.376e-06', 'epoch': '0.064'}
{'loss': '10.65', 'grad_norm': '8.438', 'learning_rate': '9.216e-06', 'epoch': '0.08'}
{'loss': '10.78', 'grad_norm': '10.19', 'learning_rate': '9.056e-06', 'epoch': '0.096'}
{'loss': '10.73', 'grad_norm': '7.938', 'learning_rate': '8.896e-06', 'epoch': '0.112'}
{'loss': '10.69', 'grad_norm': '9.812', 'learning_rate': '8.736e-06', 'epoch': '0.128'}
{'loss': '10.52', 'grad_norm': '9.625', 'learning_rate': '8.576e-06', 'epoch': '0.144'}
{'loss': '10.75', 'grad_no

## prime-anchor invariants

In [2]:
import os
import gc
import hashlib
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'nvidia/Nemotron-3.5-Bolt-E4B'
OUTPUT_DIR = "/content/drive/MyDrive/topogpt/nemotron_bolt_e4b_fineweb_topo"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]

print("Loading base model to inspect initial embedding manifold...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

if hasattr(base_model, "get_input_embeddings"):
    embed_layer = base_model.get_input_embeddings()
else:
    embed_layer = base_model.model.embed_tokens

# Transfer baseline weights to host RAM
base_embed_cpu = embed_layer.weight.data.detach().cpu().to(torch.float32)

# Load trained lm_head weights to host RAM
head_path = os.path.join(OUTPUT_DIR, "lm_head.pt")
print(f"Loading trained lm_head from {head_path}...")
state_dict = torch.load(head_path, map_location="cpu")
trained_head_cpu = state_dict["weight"].to(torch.float32)

# Clear base model from VRAM to prevent allocation overhead
del base_model
gc.collect()
torch.cuda.empty_cache()

# =====================================================================
# 1. PRIME ANCHOR AUDIT
# =====================================================================
print("\n" + "=" * 60)
print("TOPO-CBP PRIME ANCHOR AUDIT")
print("=" * 60)

max_anchor_drift = 0.0
for p in PRIME_ANCHORS:
    diff = torch.max(torch.abs(trained_head_cpu[p] - base_embed_cpu[p])).item()
    if diff > max_anchor_drift:
        max_anchor_drift = diff
    print(f"Anchor p = {p:2d} | Absolute Drift: {diff:.12f}")

anchor_tensors = [trained_head_cpu[p] for p in PRIME_ANCHORS]
serialized = torch.cat([t.flatten() for t in anchor_tensors]).numpy().tobytes()
current_hash = hashlib.sha256(serialized).hexdigest()

print("-" * 60)
print(f"Max Anchor Drift (Δ_anchor): {max_anchor_drift:.12f}")
print(f"Silicon Invariant Digest:   {current_hash}")

# =====================================================================
# 2. CHUNKED GLOBAL PLASTICITY METRICS
# =====================================================================
print("\n" + "=" * 60)
print("GLOBAL PARAMETER MANIFOLD METRICS")
print("=" * 60)

diff_matrix = trained_head_cpu - base_embed_cpu
frobenius_delta = torch.norm(diff_matrix, p='fro').item()
base_frobenius = torch.norm(base_embed_cpu, p='fro').item()
relative_frobenius = (frobenius_delta / base_frobenius) * 100.0

vocab_size = trained_head_cpu.size(0)
chunk_size = 16384
cosine_sims = []

for i in range(0, vocab_size, chunk_size):
    chunk_trained = trained_head_cpu[i:i + chunk_size]
    chunk_base = base_embed_cpu[i:i + chunk_size]
    sim = torch.nn.functional.cosine_similarity(chunk_trained, chunk_base, dim=-1)
    cosine_sims.append(sim)

all_cosine_sims = torch.cat(cosine_sims, dim=0)
mean_cosine_sim = torch.mean(all_cosine_sims).item()
min_cosine_sim = torch.min(all_cosine_sims).item()

plastic_mask = torch.ones(vocab_size, dtype=torch.bool)
for p in PRIME_ANCHORS:
    plastic_mask[p] = False

plastic_l1_drift = torch.mean(torch.abs(diff_matrix[plastic_mask])).item()

print(f"Mean Cosine Similarity:        {mean_cosine_sim:.8f}")
print(f"Minimum Token Cosine Sim:      {min_cosine_sim:.8f}")
print(f"Frobenius Norm Delta (||ΔW||): {frobenius_delta:.6f}")
print(f"Relative Frobenius Shift:      {relative_frobenius:.6f}%")
print(f"Plastic Coordinate L1 Drift:   {plastic_l1_drift:.8f}")
print("=" * 60)

Loading base model to inspect initial embedding manifold...


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Loading trained lm_head from /content/drive/MyDrive/topogpt/nemotron_bolt_e4b_fineweb_topo/lm_head.pt...

TOPO-CBP PRIME ANCHOR AUDIT
Anchor p =  2 | Absolute Drift: 0.000000000000
Anchor p =  3 | Absolute Drift: 0.000000000000
Anchor p =  5 | Absolute Drift: 0.000000000000
Anchor p =  7 | Absolute Drift: 0.000000000000
Anchor p = 11 | Absolute Drift: 0.000000000000
Anchor p = 13 | Absolute Drift: 0.000000000000
------------------------------------------------------------
Max Anchor Drift (Δ_anchor): 0.000000000000
Silicon Invariant Digest:   360af91d41ed5e98e9a368e99f51c4e1b0505fa9be973f9a46a835c379eb94b0

GLOBAL PARAMETER MANIFOLD METRICS
Mean Cosine Similarity:        0.99999577
Minimum Token Cosine Sim:      0.99896407
Frobenius Norm Delta (||ΔW||): 1.826691
Relative Frobenius Shift:      0.338550%
Plastic Coordinate L1 Drift:   0.00000882


## text generation script

In [3]:
import os
import torch
import torch.nn as nn
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

MODEL_ID = 'nvidia/Nemotron-3.5-Bolt-E4B'
OUTPUT_DIR = "/content/drive/MyDrive/topogpt/nemotron_bolt_e4b_fineweb_topo"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Load 4-bit Base Model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)
base_model.eval()

# Retrieve RMSNorm and logit soft-capping
if hasattr(base_model, "model") and hasattr(base_model.model, "norm"):
    final_norm = base_model.model.norm
elif hasattr(base_model, "norm"):
    final_norm = base_model.norm
else:
    final_norm = nn.Identity()

soft_cap = getattr(base_model.config, "final_logit_softcapping", 30.0)

# 3. Mount Certified Trained Head
hidden_size = getattr(base_model.config, "hidden_size", 2560)
vocab_size = getattr(base_model.config, "vocab_size", len(tokenizer))

lm_head = nn.Linear(hidden_size, vocab_size, bias=False, dtype=torch.bfloat16).to(DEVICE)
head_path = os.path.join(OUTPUT_DIR, "lm_head.pt")
lm_head.load_state_dict(torch.load(head_path, map_location=DEVICE))
lm_head.eval()

# Identify stop tokens
stop_token_ids = set()
if tokenizer.eos_token_id is not None:
    stop_token_ids.add(tokenizer.eos_token_id)
for token_str in ["<turn|>", "<end_of_turn>", "<|turn|>", "</s>"]:
    tid = tokenizer.convert_tokens_to_ids(token_str)
    if tid != tokenizer.unk_token_id and tid is not None:
        stop_token_ids.add(tid)

# 4. Prompt Setup
messages = [
    {
        "role": "user",
        "content": "Explain the stability-plasticity dilemma in continual neural network learning and how catastrophic forgetting can be prevented."
    }
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(DEVICE)
generated = inputs["input_ids"]

max_new_tokens = 600
min_new_tokens = 80
temperature = 0.7
top_p = 0.90
top_k = 40
no_repeat_ngram_size = 3

# 5. Generation
with torch.no_grad():
    for step in range(max_new_tokens):
        outputs = base_model(
            input_ids=generated,
            output_hidden_states=True,
            return_dict=True
        )

        last_hidden = outputs.hidden_states[-1][:, -1:, :]
        normalized_hidden = final_norm(last_hidden)
        logits = lm_head(normalized_hidden)[:, -1, :].float()

        if soft_cap is not None and soft_cap > 0:
            logits = torch.tanh(logits / soft_cap) * soft_cap

        # N-Gram Repetition Filter
        if no_repeat_ngram_size > 0 and generated.shape[1] >= no_repeat_ngram_size:
            tokens = generated[0].tolist()
            cur_ngram_prefix = tokens[-(no_repeat_ngram_size - 1):]
            banned_tokens = set()
            for i in range(len(tokens) - no_repeat_ngram_size + 1):
                if tokens[i:i + no_repeat_ngram_size - 1] == cur_ngram_prefix:
                    banned_tokens.add(tokens[i + no_repeat_ngram_size - 1])
            for b_token in banned_tokens:
                logits[0, b_token] = -float("Inf")

        if step < min_new_tokens:
            for s_id in stop_token_ids:
                logits[0, s_id] = -float("Inf")

        logits = logits / temperature

        if top_k > 0:
            indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
            logits[indices_to_remove] = -float("Inf")

        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)

        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0

        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        logits[:, indices_to_remove] = -float("Inf")

        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        generated = torch.cat([generated, next_token], dim=-1)

        if next_token.item() in stop_token_ids:
            break

response_tokens = generated[0][inputs["input_ids"].shape[1]:]
print("\n--- Model Response ---")
print(tokenizer.decode(response_tokens, skip_special_tokens=True).strip())

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]


--- Model Response ---
## The Stability-Plasticity Dilemma in Continual Neural Network Learning

The **Stability-Plasticitiy Dilemma** is a fundamental trade-off in continual (or lifelong) learning:

* **Plasticity (Learning New Tasks):** The network must be **plastic** enough to learn new tasks ($\text{Task}_2, \text{T}_3, \dots$) using new data distributions without overfitting to the old ones. This requires significant weight updates.
* **Stability (Forgetting Old Tasks):</strong> The network yet must be sufficiently **stable** to retain the knowledge ($\textsf{Task}_{1}, \textsf{\text{ }Task}_{2}, \dots, \mathrm{T}_{n-1}$) learned from previous tasks. Large, random weight updates from a new task risk overwriting crucial parameters needed for old tasks, leading to **Catastrophic Forgetting**.

**In essence:** How do you allow the network to change enough to acquire new knowledge without changing *too much* of what it already knows?

---

## Preventing Catastrophic (For)getting

Pre

## hf

In [3]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoConfig

MODEL_ID = 'nvidia/Nemotron-3.5-Bolt-E4B'
OUTPUT_DIR = "/content/drive/MyDrive/topogpt/nemotron_bolt_e4b_fineweb_topo"
HF_EXPORT_DIR = "/content/drive/MyDrive/topogpt/nemotron_bolt_e4b_fineweb_topo_hf"
os.makedirs(HF_EXPORT_DIR, exist_ok=True)

print(f"Exporting model artifacts to Hugging Face format at: {HF_EXPORT_DIR}")

# 1. Save Tokenizer
print("Exporting tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)
tokenizer.save_pretrained(HF_EXPORT_DIR)

# 2. Save Model Configuration
print(f"Exporting architecture configuration for {MODEL_ID}...")
config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
config.save_pretrained(HF_EXPORT_DIR)

# 3. Export Trained LM Head Weights
head_path = os.path.join(OUTPUT_DIR, "lm_head.pt")
print(f"Loading trained weights from: {head_path}")
state_dict = torch.load(head_path, map_location="cpu")
torch.save(state_dict, os.path.join(HF_EXPORT_DIR, "lm_head.pt"))

# 4. Generate Topo-CBP Audit Metadata
metadata = {
    "base_model": MODEL_ID,
    "framework": "Topological Continual Backpropagation (Topo-CBP)",
    "prime_anchors": [2, 3, 5, 7, 11, 13],
    "safety_constant_lambda": 0.9785142874,
    "max_anchor_drift": 0.000000000000,
    "certified_sha256_digest": "360af91d41ed5e98e9a368e99f51c4e1b0505fa9be973f9a46a835c379eb94b0",
    "dataset": "HuggingFaceFW/fineweb (sample-10BT)",
    "quantization": "4-bit NF4 bedrock",
    "status": "CERTIFIED_BITWISE_INVARIANT"
}

with open(os.path.join(HF_EXPORT_DIR, "topo_cbp_receipt.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("\n--- Export Summary ---")
print(f"Destination: {HF_EXPORT_DIR}")
print("Files packaged:")
for fname in os.listdir(HF_EXPORT_DIR):
    fsize = os.path.getsize(os.path.join(HF_EXPORT_DIR, fname)) / (1024 * 1024)
    print(f"  • {fname:25s} ({fsize:.2f} MB)")

Exporting model artifacts to Hugging Face format at: /content/drive/MyDrive/topogpt/nemotron_bolt_e4b_fineweb_topo_hf
Exporting tokenizer...
Exporting architecture configuration for nvidia/Nemotron-3.5-Bolt-E4B...
Loading trained weights from: /content/drive/MyDrive/topogpt/nemotron_bolt_e4b_fineweb_topo/lm_head.pt

--- Export Summary ---
Destination: /content/drive/MyDrive/topogpt/nemotron_bolt_e4b_fineweb_topo_hf
Files packaged:
  • chat_template.jinja       (0.02 MB)
  • tokenizer_config.json     (0.00 MB)
  • tokenizer.json            (30.68 MB)
  • config.json               (0.00 MB)
  • lm_head.pt                (1280.00 MB)
  • topo_cbp_receipt.json     (0.00 MB)


In [4]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, create_repo

# Retrieve credentials and set target repository
HF_TOKEN = userdata.get('HF_TOKEN')
user_or_name = 'frankmorales2020'
model_name = "nemotron-3.5-bolt-e4b-topo-governed-fineweb"
repo_id = f"{user_or_name}/{model_name}"

EXPORT_DIR = "/content/drive/MyDrive/topogpt/nemotron_bolt_e4b_fineweb_topo_hf"

api = HfApi(token=HF_TOKEN)

print(f"Creating/verifying Hugging Face repository: {repo_id}...")
create_repo(
    repo_id=repo_id,
    token=HF_TOKEN,
    repo_type="model",
    exist_ok=True
)

print(f"Uploading artifacts from {EXPORT_DIR} to {repo_id}...")
api.upload_folder(
    folder_path=EXPORT_DIR,
    repo_id=repo_id,
    repo_type="model",
    token=HF_TOKEN,
    commit_message="Release certified Nemotron-3.5-Bolt-E4B Topo-CBP governed head on FineWeb (Delta_drift=0)"
)

print("\nModel uploaded successfully.")
print(f"Repository URL: https://huggingface.co/{repo_id}")

Creating/verifying Hugging Face repository: frankmorales2020/nemotron-3.5-bolt-e4b-topo-governed-fineweb...
Uploading artifacts from /content/drive/MyDrive/topogpt/nemotron_bolt_e4b_fineweb_topo_hf to frankmorales2020/nemotron-3.5-bolt-e4b-topo-governed-fineweb...

Model uploaded successfully.
Repository URL: https://huggingface.co/frankmorales2020/nemotron-3.5-bolt-e4b-topo-governed-fineweb


## inference-hf

In [5]:
import os
import torch
import torch.nn as nn
from huggingface_hub import hf_hub_download
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

REPO_ID = "frankmorales2020/nemotron-3.5-bolt-e4b-topo-governed-fineweb"
BASE_ID = "nvidia/Nemotron-3.5-Bolt-E4B"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =====================================================================
# 1. LOAD TOKENIZER DIRECTLY FROM YOUR HF REPOSITORY
# =====================================================================
print(f"Loading tokenizer from: {REPO_ID}...")
tokenizer = AutoTokenizer.from_pretrained(REPO_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =====================================================================
# 2. LOAD QUANTIZED BASE MODEL (4-BIT NF4)
# =====================================================================
print(f"Loading base model: {BASE_ID} in 4-bit precision...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)
base_model.eval()

# Locate RMSNorm layer
if hasattr(base_model, "model") and hasattr(base_model.model, "norm"):
    final_norm = base_model.model.norm
elif hasattr(base_model, "norm"):
    final_norm = base_model.norm
else:
    final_norm = nn.Identity()

soft_cap = getattr(base_model.config, "final_logit_softcapping", 30.0)

# =====================================================================
# 3. DOWNLOAD & MOUNT TRAINED LM_HEAD FROM YOUR HF REPO
# =====================================================================
print(f"Downloading trained lm_head.pt from {REPO_ID}...")
head_file = hf_hub_download(repo_id=REPO_ID, filename="lm_head.pt")

hidden_size = getattr(base_model.config, "hidden_size", 2560)
vocab_size = getattr(base_model.config, "vocab_size", len(tokenizer))

lm_head = nn.Linear(hidden_size, vocab_size, bias=False, dtype=torch.bfloat16).to(DEVICE)
lm_head.load_state_dict(torch.load(head_file, map_location=DEVICE))
lm_head.eval()

# Configure turn termination tokens
stop_token_ids = set()
if tokenizer.eos_token_id is not None:
    stop_token_ids.add(tokenizer.eos_token_id)
for token_str in ["<turn|>", "<end_of_turn>", "<|turn|>", "</s>"]:
    tid = tokenizer.convert_tokens_to_ids(token_str)
    if tid != tokenizer.unk_token_id and tid is not None:
        stop_token_ids.add(tid)

# =====================================================================
# 4. STRUCTURED CHAT PROMPT
# =====================================================================
messages = [
    {
        "role": "user",
        "content": "Explain the stability-plasticity dilemma in continual neural network learning and how catastrophic forgetting can be prevented."
    }
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(f"\nPrompt formatted:\n{formatted_prompt}\nGenerating response...\n")

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(DEVICE)
generated = inputs["input_ids"]

max_new_tokens = 600
min_new_tokens = 60
temperature = 0.7
top_p = 0.90
top_k = 40
no_repeat_ngram_size = 3

# =====================================================================
# 5. AUTOREGRESSIVE GENERATION LOOP
# =====================================================================
with torch.no_grad():
    for step in range(max_new_tokens):
        outputs = base_model(
            input_ids=generated,
            output_hidden_states=True,
            return_dict=True
        )

        last_hidden = outputs.hidden_states[-1][:, -1:, :]
        normalized_hidden = final_norm(last_hidden)
        logits = lm_head(normalized_hidden)[:, -1, :].float()

        # Gemma logit soft-capping
        if soft_cap is not None and soft_cap > 0:
            logits = torch.tanh(logits / soft_cap) * soft_cap

        # Strict N-gram blocking to avoid loops
        if no_repeat_ngram_size > 0 and generated.shape[1] >= no_repeat_ngram_size:
            tokens = generated[0].tolist()
            cur_ngram_prefix = tokens[-(no_repeat_ngram_size - 1):]
            banned_tokens = set()
            for i in range(len(tokens) - no_repeat_ngram_size + 1):
                if tokens[i:i + no_repeat_ngram_size - 1] == cur_ngram_prefix:
                    banned_tokens.add(tokens[i + no_repeat_ngram_size - 1])
            for b_token in banned_tokens:
                logits[0, b_token] = -float("Inf")

        if step < min_new_tokens:
            for s_id in stop_token_ids:
                logits[0, s_id] = -float("Inf")

        logits = logits / temperature

        if top_k > 0:
            indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
            logits[indices_to_remove] = -float("Inf")

        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)

        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0

        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        logits[:, indices_to_remove] = -float("Inf")

        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        generated = torch.cat([generated, next_token], dim=-1)

        if next_token.item() in stop_token_ids:
            break

# Decode newly generated tokens
response_tokens = generated[0][inputs["input_ids"].shape[1]:]
output_text = tokenizer.decode(response_tokens, skip_special_tokens=True)

print("--- Model Response ---")
print(output_text.strip())

Loading tokenizer from: frankmorales2020/nemotron-3.5-bolt-e4b-topo-governed-fineweb...


config.json:   0%|          | 0.00/4.73k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.84k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Loading base model: nvidia/Nemotron-3.5-Bolt-E4B in 4-bit precision...


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

lm_head.pt: reconstructing file:   0%|          |  0.00B / 1.34GB            

lm_head.pt: downloading bytes:           |  0.00B            


Prompt formatted:
<bos><|turn>user
Explain the stability-plasticity dilemma in continual neural network learning and how catastrophic forgetting can be prevented.<turn|>
<|turn>model

Generating response...

--- Model Response ---
## The Stability-Plasticity Dilemma in Continual Neural Network Learning

The **Stability-Plasticitys Dilemma** is a fundamental and persistent challenge in **Continual Learning (Lifelong Learning)**. It describes the inherent tension between two necessary but opposing requirements for a neural network to learn sequentially across multiple tasks ($\text{Task}_1, \text{then } \textsf{Task}\_2, \dots, \rightarrow \text sf{\textsf{.Task}_T}}$):

1. **Stability (Retention):** The network must retain the knowledge (weights and internal representations) learned from **previous tasks** ($\textsf{\text{Retention}(\textsfTask}_i \rightarrow i+1)}$) with high fidelity. If it forgets $\textsf Task}_1$ while learning $\text sfTask}_2$, the continual learning process has

In [2]:
import torch
import torch.nn as nn
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

REPO_ID = "frankmorales2020/nemotron-3.5-bolt-e4b-topo-governed-fineweb"
BASE_ID = "nvidia/Nemotron-3.5-Bolt-E4B"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load Tokenizer & 4-bit Base Model
tokenizer = AutoTokenizer.from_pretrained(REPO_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    trust_remote_code=True
)
base_model.eval()

# Retrieve RMSNorm
if hasattr(base_model, "model") and hasattr(base_model.model, "norm"):
    final_norm = base_model.model.norm
elif hasattr(base_model, "norm"):
    final_norm = base_model.norm
else:
    final_norm = nn.Identity()

# 2. Download and Mount Trained Head
head_file = hf_hub_download(repo_id=REPO_ID, filename="lm_head.pt")
lm_head = nn.Linear(2560, len(tokenizer), bias=False, dtype=torch.bfloat16).to(DEVICE)
lm_head.load_state_dict(torch.load(head_file, map_location=DEVICE))
lm_head.eval()

# 3. Generation Function
def generate(prompt: str, max_new_tokens: int = 512):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(DEVICE)
    generated = inputs["input_ids"]

    stop_tokens = {tokenizer.eos_token_id}
    for t_str in ["<turn|>", "<end_of_turn>", "<|turn|>"]:
        tid = tokenizer.convert_tokens_to_ids(t_str)
        if tid != tokenizer.unk_token_id and tid is not None:
            stop_tokens.add(tid)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            out = base_model(input_ids=generated, output_hidden_states=True, return_dict=True)
            hidden = final_norm(out.hidden_states[-1][:, -1:, :])
            logits = lm_head(hidden)[:, -1, :].float()

            logits = logits / 0.7
            probs = torch.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            generated = torch.cat([generated, next_tok], dim=-1)

            if next_tok.item() in stop_tokens:
                break

    response_ids = generated[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(response_ids, skip_special_tokens=True).strip()

print(generate("Explain how topological gradient locking prevents catastrophic forgetting."))

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Topological gradient locking prevents catastrophic forgetting by **constraining the learning process to remain within a specific, stable topological manifold of the model's parameter space, thereby preserving the geometric relationships (topological invariants) associated with previously learned tasks.**

Here is a detailed breakdown of how this mechanism works:

---

### 1. The Problem: Catastrophic Forgetting

In standard neural network fine-tuning (e.g., transfer learning), when the model is trained on a new task ($\text{Task B}$), the gradient updates ($\Delta \theta$) are often large and aggressive. These updates can drastically alter the weights ($\theta$) in a way that destroys the specific, low-dimensional geometric structures (like decision boundaries or latent representations) learned for the old task ($\text{Task A}$). The model "forgets" $\text{Task A}$ because the parameter configuration that supported $\text{Task A}$ is no longer reachable or stable under the new loss lan